# Read Excel and Build Bilingual Dictionaries

本筆記本會讀取指定 Excel，建立中英文字典並指派到變數。
This notebook reads the specified Excel file, builds Chinese/English dictionaries, and assigns them to variables.

## 1) Import Libraries and Define File Path
匯入套件並設定 Excel 路徑。

In [5]:
import pandas as pd
import json

file_path = r"C:\Users\ian.leong\OneDrive - Fubon Financial Holding Co., Ltd\桌面\新標籤特徵.xlsx"
file_path

'C:\\Users\\ian.leong\\OneDrive - Fubon Financial Holding Co., Ltd\\桌面\\新標籤特徵.xlsx'

## 2) Load Excel Sheet into DataFrame
讀取 Excel，並先檢視前幾列與欄位。

In [6]:
# If you need a specific sheet, replace with: pd.read_excel(file_path, sheet_name="Sheet1")
df = pd.read_excel(file_path)

print("Data preview:")
display(df.head())

print("\nColumns:")
print(df.columns.tolist())

Data preview:


,表格名稱,特徵,類別,PK,資料當責單位(業務端),欄位規則依據,Business Glossory,Business Rule,異動別
0,CF_CUSTID,CUSTOMER_ID,VARCHAR2(32),V,數據科學部,3-商業邏輯,客戶 ID,NaN,新增
1,CF_CUSTID,YYYYMM,VARCHAR2(6),V,數據科學部,3-商業邏輯,特徵年月,NaN,新增
2,CF_TXN_AP,CUSTOMER_ID,VARCHAR2(32),V,數據科學部,3-商業邏輯,客戶 ID,NaN,新增
3,CF_TXN_AP,YYYYMM,VARCHAR2(6),V,數據科學部,3-商業邏輯,特徵年月,NaN,新增
4,CF_TXN_AP,AP_BUY_AMT_12M,"NUMBER(18,0)",NaN,數據科學部,3-商業邏輯,所有商品年買入金額,計算客戶一年內所有商品買入之金額,新增



Columns:
['表格名稱', '特徵', '類別', 'PK', '資料當責單位(業務端)', '欄位規則依據', 'Business Glossory', 'Business Rule', '異動別']


## 3) Inspect Columns and Select Chinese/English Fields
自動偵測中英文欄位；找不到時預設前兩欄。

In [11]:
# Force column mapping based on your rule:
# Chinese  -> Business Glossory
# English  -> 特徵
zh_col = "Business Glossory"
en_col = "特徵"

missing = [c for c in [zh_col, en_col] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}. Available columns: {df.columns.tolist()}")

print(f"Selected Chinese column: {zh_col}")
print(f"Selected English column: {en_col}")

# Keep only valid rows, trim whitespace, and lowercase English values
work_df = df[[zh_col, en_col]].copy()
work_df[zh_col] = work_df[zh_col].astype(str).str.strip()
work_df[en_col] = work_df[en_col].astype(str).str.strip().str.lower()
work_df = work_df[(work_df[zh_col] != "") & (work_df[en_col] != "")]
work_df = work_df[~work_df[zh_col].str.lower().eq("nan") & ~work_df[en_col].str.lower().eq("nan")]

print(f"Valid rows for mapping: {len(work_df)}")

Selected Chinese column: Business Glossory
Selected English column: 特徵
Valid rows for mapping: 4217


## 4) Build Dictionaries and Assign Variables
建立雙向 dic，並 assign 到單一 var 方便後續使用。

In [12]:
# Later duplicates overwrite earlier ones (last value wins)
zh_to_en_dic = dict(zip(work_df[zh_col], work_df[en_col]))
en_to_zh_dic = dict(zip(work_df[en_col], work_df[zh_col]))

# Assign to reusable variables
dic_zh_en = zh_to_en_dic
dic_en_zh = en_to_zh_dic

# One combined variable as requested
bilingual_dic = {
    "zh_to_en": zh_to_en_dic,
    "en_to_zh": en_to_zh_dic,
}

print(f"zh_to_en_dic size: {len(zh_to_en_dic)}")
print(f"en_to_zh_dic size: {len(en_to_zh_dic)}")

print("\nSample zh_to_en_dic (first 10):")
for i, (k, v) in enumerate(zh_to_en_dic.items()):
    if i >= 10:
        break
    print(f"{k} -> {v}")

print("\nSample en_to_zh_dic (first 10):")
for i, (k, v) in enumerate(en_to_zh_dic.items()):
    if i >= 10:
        break
    print(f"{k} -> {v}")

zh_to_en_dic size: 4104
en_to_zh_dic size: 4125

Sample zh_to_en_dic (first 10):
客戶 ID -> customer_id
特徵年月 -> yyyymm
所有商品年買入金額 -> ap_buy_amt_12m
所有商品年買入金額增減 -> ap_buy_amt_12m_mdiff
所有商品年買入金額增減比例 -> ap_buy_amt_12m_rdiff
所有商品季買入金額 -> ap_buy_amt_3m
所有商品季買入金額增減 -> ap_buy_amt_3m_mdiff
所有商品季買入金額增減比例 -> ap_buy_amt_3m_rdiff
所有商品半年買入金額 -> ap_buy_amt_6m
所有商品半年買入金額增減 -> ap_buy_amt_6m_mdiff

Sample en_to_zh_dic (first 10):
customer_id -> 客戶 ID
yyyymm -> 特徵年月
ap_buy_amt_12m -> 所有商品年買入金額
ap_buy_amt_12m_mdiff -> 所有商品年買入金額增減
ap_buy_amt_12m_rdiff -> 所有商品年買入金額增減比例
ap_buy_amt_3m -> 所有商品季買入金額
ap_buy_amt_3m_mdiff -> 所有商品季買入金額增減
ap_buy_amt_3m_rdiff -> 所有商品季買入金額增減比例
ap_buy_amt_6m -> 所有商品半年買入金額
ap_buy_amt_6m_mdiff -> 所有商品半年買入金額增減


In [13]:
en_to_zh_dic

{'customer_id': '客戶 ID',
 'yyyymm': '特徵年月',
 'ap_buy_amt_12m': '所有商品年買入金額',
 'ap_buy_amt_12m_mdiff': '所有商品年買入金額增減',
 'ap_buy_amt_12m_rdiff': '所有商品年買入金額增減比例',
 'ap_buy_amt_3m': '所有商品季買入金額',
 'ap_buy_amt_3m_mdiff': '所有商品季買入金額增減',
 'ap_buy_amt_3m_rdiff': '所有商品季買入金額增減比例',
 'ap_buy_amt_6m': '所有商品半年買入金額',
 'ap_buy_amt_6m_mdiff': '所有商品半年買入金額增減',
 'ap_buy_amt_6m_rdiff': '所有商品半年買入金額增減比例',
 'ap_buy_amt_m': '所有商品月買入金額',
 'ap_buy_amt_m_mdiff': '所有商品月買入金額增減',
 'ap_buy_amt_m_rdiff': '所有商品月買入金額增減比例',
 'ap_buy_day_12m': '所有商品年買入天數',
 'ap_buy_day_12m_mdiff': '所有商品年買入天數增減',
 'ap_buy_day_12m_rdiff': '所有商品年買入天數增減比例',
 'ap_buy_day_3m': '所有商品季買入天數',
 'ap_buy_day_3m_mdiff': '所有商品季買入天數增減',
 'ap_buy_day_3m_rdiff': '所有商品季買入天數增減比例',
 'ap_buy_day_6m': '所有商品半年買入天數',
 'ap_buy_day_6m_mdiff': '所有商品半年買入天數增減',
 'ap_buy_day_6m_rdiff': '所有商品半年買入天數增減比例',
 'ap_buy_day_m': '所有商品月買入天數',
 'ap_buy_day_m_mdiff': '所有商品月買入天數增減',
 'ap_buy_day_m_rdiff': '所有商品月買入天數增減比例',
 'ap_buy_prod_12m': '所有商品年買入次數',
 'ap_buy_prod_12m_mdiff': '所有

## 5) Optional: Export Dictionaries to JSON
可選：將 dic 輸出成 JSON（保留中文可讀性）。

In [ ]:
# Optional export
# with open("dic_zh_en.json", "w", encoding="utf-8") as f:
#     json.dump(dic_zh_en, f, ensure_ascii=False, indent=2)
#
# with open("dic_en_zh.json", "w", encoding="utf-8") as f:
#     json.dump(dic_en_zh, f, ensure_ascii=False, indent=2)

# Quick lookup examples
print("\nLookup example:")
print("dic_zh_en.get('範例中文') =>", dic_zh_en.get("範例中文"))
print("dic_en_zh.get('sample_english') =>", dic_en_zh.get("sample_english"))